In [1]:
#start dpo

In [1]:
# 1. 移除所有舊版本
!pip uninstall -y transformers tokenizers huggingface_hub accelerate peft bitsandbytes safetensors datasets trl

# 2. 清除 Cache
!rm -rf /opt/conda/lib/python3.11/site-packages/transformers*
!rm -rf ~/.cache/huggingface

# 3. 安裝乾淨的新版本（與 Mistral / Llama / Qwen / DPO 完全相容）
!pip install -U --no-cache-dir \
    "transformers>=4.46.2" \
    "trl>=0.10.1" \
    "peft>=0.12.0" \
    "accelerate>=0.33.0" \
    "datasets>=2.19.0" \
    bitsandbytes \
    safetensors
print(1)

Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
Found existing installation: tokenizers 0.22.1
Uninstalling tokenizers-0.22.1:
  Successfully uninstalled tokenizers-0.22.1
Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Successfully uninstalled huggingface-hub-0.36.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.0
Uninstalling peft-0.18.0:
  Successfully uninstalled peft-0.18.0
Found existing installation: bitsandbytes 0.48.2
Uninstalling bitsandbytes-0.48.2:
  Successfully uninstalled bitsandbytes-0.48.2
Found existing installation: safetensors 0.7.0
Uninstalling safetensors-0.7.0:
  Successfully uninstalled safetensors-0.7.0
Found existing installation: datasets 4.4.1
Uninstalling datasets-4.4.1:
  Successfully uninstalled datasets-4.4.1
Found ex

In [3]:
##use this to check your memory
!nvidia-smi

Wed Dec  3 19:02:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   22C    P8             16W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1️⃣ 套件
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import DPOTrainer

2025-12-03 19:03:06.400998: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764788586.419163   11592 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764788586.425131   11592 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-03 19:03:06.457171: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
# 2️⃣ 讀資料
# dataset 檔案範例為 jsonl，每行格式：
# {"prompt": "question...", "chosen": "good answer...", "rejected": "bad answer..."}
dataset = load_dataset("json", data_files="updated_dpo_finetune.jsonl")

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
import torch
# use bf16 is great on A10G and fallback to fp16 
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

In [2]:
!nvidia-smi

Wed Dec  3 19:00:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   26C    P0             59W /  300W |   14453MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(BASE, use_fast=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE,
    device_map="auto",
    torch_dtype="auto",
    load_in_4bit=True,   # 視 GPU 調整
)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [6]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "./bio_mistral_lora",   # 你的原始 LoRA
)
model.print_trainable_parameters()


trainable params: 0 || all params: 7,283,675,136 || trainable%: 0.0000


In [9]:
from trl import DPOTrainer, DPOConfig

training_args = DPOConfig(
    output_dir="./dpo-out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    num_train_epochs=1,
    save_strategy="epoch",
    logging_steps=50,
    beta=0.1,
    fp16=True,
    gradient_checkpointing=True,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    train_dataset=dataset["train"],
    eval_dataset=dataset.get("validation"),
    args=training_args,
)

trainer.train()

trainer.model.save_pretrained("./dpo-finetuned2")
tokenizer.save_pretrained("./dpo-finetuned2")


/opt/conda/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
50,0.151600


('./dpo-finetuned2/tokenizer_config.json',
 './dpo-finetuned2/special_tokens_map.json',
 './dpo-finetuned2/chat_template.jinja',
 './dpo-finetuned2/tokenizer.model',
 './dpo-finetuned2/added_tokens.json',
 './dpo-finetuned2/tokenizer.json')

In [4]:
# dpo model merge

SyntaxError: invalid syntax (2063261137.py, line 2)

In [1]:
# 1️⃣ 卸載舊版套件，避免版本衝突
!pip uninstall -y transformers peft trl accelerate bitsandbytes datasets sentencepiece protobuf

# 2️⃣ 安裝相容版本
!pip install transformers==4.35.0
!pip install peft==0.5.0
!pip install trl==0.7.0
!pip install accelerate==0.22.0
!pip install bitsandbytes datasets sentencepiece protobuf
!pip install accelerate==0.22.0


Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
Found existing installation: peft 0.5.0
Uninstalling peft-0.5.0:
  Successfully uninstalled peft-0.5.0
Found existing installation: trl 0.7.0
Uninstalling trl-0.7.0:
  Successfully uninstalled trl-0.7.0
Found existing installation: accelerate 0.22.0
Uninstalling accelerate-0.22.0:
  Successfully uninstalled accelerate-0.22.0
Found existing installation: bitsandbytes 0.48.2
Uninstalling bitsandbytes-0.48.2:
  Successfully uninstalled bitsandbytes-0.48.2
Found existing installation: datasets 4.4.1
Uninstalling datasets-4.4.1:
  Successfully uninstalled datasets-4.4.1
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1
Found existing installation: protobuf 6.33.1
Uninstalling protobuf-6.33.1:
  Successfully uninstalled protobuf-6.33.1
  Using cached transformers-4.35.0-py3-none-any.wh

In [10]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ====== 設定路徑 ======
BASE       = "BioMistral/BioMistral-7B"
DPO_ADAPTER = "./dpo-finetuned2"
MERGED_OUT  = "bio_mistral_dpo_merged2"

# ====== 1️⃣ 載入 tokenizer ======
tok = AutoTokenizer.from_pretrained(BASE, use_fast=True)

# ====== 2️⃣ 載入 base 模型 ======
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
).to(device)  # 直接搬到 GPU

# ====== 3️⃣ 載入 DPO LoRA adapter 並 merge ======
model = PeftModel.from_pretrained(model, DPO_ADAPTER)
with torch.no_grad():
    model = model.merge_and_unload()

# ====== 4️⃣ 儲存合併後模型 ======
os.makedirs(MERGED_OUT, exist_ok=True)
model.save_pretrained(MERGED_OUT, safe_serialization=True)
tok.save_pretrained(MERGED_OUT)

print("✅ DPO merged model saved to:", MERGED_OUT)


`torch_dtype` is deprecated! Use `dtype` instead!


✅ DPO merged model saved to: bio_mistral_dpo_merged2
